In [1]:
import sqlite3
import datetime

# ---------------- Database Setup ----------------
conn = sqlite3.connect("blood_bank.db")
cursor = conn.cursor()

# Donors table
cursor.execute("""
CREATE TABLE IF NOT EXISTS donors (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    age INTEGER,
    blood_group TEXT,
    contact TEXT,
    last_donation DATE
)
""")

# Inventory table
cursor.execute("""
CREATE TABLE IF NOT EXISTS inventory (
    blood_group TEXT PRIMARY KEY,
    units INTEGER
)
""")

# Initialize inventory with all blood groups if not present
blood_groups = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]
for bg in blood_groups:
    cursor.execute("INSERT OR IGNORE INTO inventory (blood_group, units) VALUES (?, ?)", (bg, 0))
conn.commit()

# ---------------- Functions ----------------
def add_donor():
    name = input("Enter donor name: ")
    age = int(input("Enter age: "))
    blood_group = input("Enter blood group (A+/O-/...): ").upper()
    contact = input("Enter contact number: ")

    if age < 18:
        print(" Donor must be at least 18 years old.")
        return

    cursor.execute("INSERT INTO donors (name, age, blood_group, contact, last_donation) VALUES (?, ?, ?, ?, ?)",
                   (name, age, blood_group, contact, None))
    conn.commit()
    print(" Donor registered successfully!")

def record_donation():
    donor_id = int(input("Enter donor ID: "))
    units = int(input("Enter units donated: "))

    cursor.execute("SELECT blood_group, last_donation FROM donors WHERE id=?", (donor_id,))
    donor = cursor.fetchone()
    if not donor:
        print(" Donor not found.")
        return

    blood_group, last_donation = donor
    if last_donation:
        last_date = datetime.datetime.strptime(last_donation, "%Y-%m-%d")
        if (datetime.datetime.now() - last_date).days < 90:
            print(" Donor must wait 3 months before donating again.")
            return

    # Update inventory
    cursor.execute("UPDATE inventory SET units = units + ? WHERE blood_group=?", (units, blood_group))
    # Update donor last donation date
    cursor.execute("UPDATE donors SET last_donation=? WHERE id=?", (datetime.datetime.now().strftime("%Y-%m-%d"), donor_id))
    conn.commit()
    print(" Donation recorded successfully!")

def request_blood():
    blood_group = input("Enter required blood group: ").upper()
    units = int(input("Enter units required: "))

    cursor.execute("SELECT units FROM inventory WHERE blood_group=?", (blood_group,))
    available = cursor.fetchone()[0]

    if available >= units:
        cursor.execute("UPDATE inventory SET units = units - ? WHERE blood_group=?", (units, blood_group))
        conn.commit()
        print(f" Request fulfilled. {units} units of {blood_group} provided.")
    else:
        print(f" Not enough stock. Available units: {available}")

def search_donor():
    blood_group = input("Enter blood group to search: ").upper()
    cursor.execute("SELECT id, name, age, contact FROM donors WHERE blood_group=?", (blood_group,))
    donors = cursor.fetchall()
    if donors:
        print(" Matching Donors:")
        for d in donors:
            print(f"ID: {d[0]}, Name: {d[1]}, Age: {d[2]}, Contact: {d[3]}")
    else:
        print(" No donors found.")

def view_inventory():
    cursor.execute("SELECT * FROM inventory")
    print(" Blood Inventory:")
    for row in cursor.fetchall():
        print(f"{row[0]} : {row[1]} units")

# ---------------- Menu ----------------
def menu():
    while True:
        print("\n=== Blood Donation Management System ===")
        print("1. Register Donor")
        print("2. Record Donation")
        print("3. Request Blood")
        print("4. Search Donor")
        print("5. View Inventory")
        print("6. Exit")

        choice = input("Enter choice: ")
        if choice == "1":
            add_donor()
        elif choice == "2":
            record_donation()
        elif choice == "3":
            request_blood()
        elif choice == "4":
            search_donor()
        elif choice == "5":
            view_inventory()
        elif choice == "6":
            print("Exiting system...")
            break
        else:
            print(" Invalid choice. Try again.")

# ---------------- Run ----------------
menu()
conn.close()



=== Blood Donation Management System ===
1. Register Donor
2. Record Donation
3. Request Blood
4. Search Donor
5. View Inventory
6. Exit
Enter choice: 1
Enter donor name: premal
Enter age: 18
Enter blood group (A+/O-/...): O+
Enter contact number: 9322684811
 Donor registered successfully!

=== Blood Donation Management System ===
1. Register Donor
2. Record Donation
3. Request Blood
4. Search Donor
5. View Inventory
6. Exit
Enter choice: 2
Enter donor ID: 1
Enter units donated: 2
 Donation recorded successfully!

=== Blood Donation Management System ===
1. Register Donor
2. Record Donation
3. Request Blood
4. Search Donor
5. View Inventory
6. Exit
Enter choice: 3
Enter required blood group: o+
Enter units required: 1
 Request fulfilled. 1 units of O+ provided.

=== Blood Donation Management System ===
1. Register Donor
2. Record Donation
3. Request Blood
4. Search Donor
5. View Inventory
6. Exit
Enter choice: 4
Enter blood group to search: o+
 Matching Donors:
ID: 1, Name: premal, Ag

# **How This Works**

- **Register Donor** → Adds donor info into database.

- **Record Donation** → Updates donor’s last donation date and adds units to inventory.

- **Request Blood** → Deducts units from inventory if available.

- **Search Donor** → Finds donors by blood group.

- **View Inventory** → Shows current stock of all blood groups.

## **Right now, the program is waiting for you to choose an option from the menu.**

# **For example:**

- **If you type 1** → it will ask you to enter donor details (name, age, blood group, contact).

- **If you type 2** → it will record a donation for an existing donor.

- **If you type 3**→ it will let you request blood units from the inventory.

- **If you type 4** → it will search donors by blood group.

- **If you type 5** → it will show the current blood stock.

- **If you type 6** → it will exit the program.

- Try typing 1 and pressing Enter — that will start the Register Donor process.